# Hugging Face LLM Course — Chapter 3.1 Notebook  
## Introduction to Fine-tuning a Pretrained Model

This notebook is adapted for classroom use from Hugging Face LLM Course Chapter 3.1:  
**Fine-tuning a pretrained model — Introduction**

Original chapter: https://huggingface.co/learn/llm-course/chapter3/1

### Learning goals

By the end of this notebook, students should be able to explain:

1. Why we fine-tune a pretrained Transformer model.
2. What libraries are involved in the Hugging Face fine-tuning workflow.
3. The difference between using a pretrained model directly and adapting it to a downstream task.
4. The basic workflow of fine-tuning for text classification.
5. Why PyTorch, Datasets, Transformers, Evaluate, and Accelerate are commonly used together.

## 1. Big Picture: Why Fine-tune?

In earlier Transformer examples, we often used a pretrained model directly:

```python
pipeline("sentiment-analysis")
```

That is convenient, but sometimes we need the model to solve **our own task**, such as:

- Classifying IMDb movie reviews as positive/negative
- Detecting spam emails
- Classifying student feedback
- Categorizing support tickets
- Identifying toxic comments
- Classifying news articles by topic

Fine-tuning means:

> Start from a pretrained model and continue training it on a smaller task-specific dataset.

We are **not training the whole Transformer from scratch**.  
We are adapting an existing model to a specific task.

## 2. The Core Hugging Face Fine-tuning Stack

This chapter introduces several Hugging Face libraries:

| Library | Role |
|---|---|
| `transformers` | Provides pretrained models, tokenizers, pipelines, and Trainer API |
| `datasets` | Loads and processes datasets efficiently |
| `evaluate` | Computes metrics such as accuracy, F1, precision, recall |
| `accelerate` | Helps run training on CPU, GPU, TPU, or distributed devices |
| PyTorch | The deep learning framework used for model training |

For teaching, the simplest workflow is:

```text
dataset → tokenizer → model → Trainer → evaluation → save/share model
```

## 3. Install Required Packages

For Google Colab, run the following cell first.

After installation, it is sometimes useful to restart the runtime:

```text
Runtime → Restart runtime
```

Then run the notebook again from the top.

In [ ]:
!pip install -q transformers==4.41.2 datasets==2.19.1 evaluate==0.4.2 accelerate==0.30.1

## 4. Check Library Versions

In [ ]:
import transformers
import datasets
import evaluate
import accelerate

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("accelerate:", accelerate.__version__)

## 5. A Simple Motivating Example

Before fine-tuning, let us load a general pretrained model checkpoint.

We will use:

```text
distilbert-base-uncased
```

This is a smaller version of BERT. It is suitable for classroom examples because it is faster and lighter than full BERT.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

### Important note

The warning about newly initialized classification weights is expected.

Why?

`distilbert-base-uncased` is a pretrained language representation model.  
It was not originally trained specifically for IMDb positive/negative sentiment classification.

When we load it with:

```python
AutoModelForSequenceClassification
```

Hugging Face adds a new classification head on top of DistilBERT.

That classification head must be trained on our task-specific dataset.

## 6. Tokenization Example

Transformer models do not directly read raw text.  
They read token IDs.

Let us convert a movie review sentence into model input.

In [ ]:
text = "This movie was surprisingly touching and beautifully acted."

encoded = tokenizer(text, return_tensors="pt")

print(encoded.keys())
print("input_ids shape:", encoded["input_ids"].shape)
print("attention_mask shape:", encoded["attention_mask"].shape)
print("First few token IDs:", encoded["input_ids"][0][:10])

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print(tokens)

## 7. Load a Dataset from the Hub

In later sections of Chapter 3, Hugging Face shows how to prepare data and fine-tune a model.

Here, we preview the workflow using the IMDb dataset.

For classroom speed, we will use only a small subset.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

dataset

In [ ]:
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(small_train_dataset)
print(small_eval_dataset)

## 8. Inspect the Dataset

IMDb labels are usually:

```text
0 = negative
1 = positive
```

In [ ]:
example = small_train_dataset[0]

print("Text:")
print(example["text"][:500])
print()
print("Label:", example["label"])

## 9. Preprocess the Dataset

We tokenize all text examples.

To keep the example fast and stable on Colab, we use:

```text
max_length = 128
```

For real training, you may use a larger max length, but it requires more memory.

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_eval = small_eval_dataset.map(tokenize_function, batched=True)

tokenized_train

## 10. What Will Fine-tuning Do?

Before fine-tuning, the classification head is mostly random.

During fine-tuning:

1. The tokenized IMDb reviews are passed into DistilBERT.
2. DistilBERT produces contextual text representations.
3. The classification head maps those representations to two labels.
4. The loss compares predictions with IMDb labels.
5. Backpropagation updates the model weights.

The goal is to adapt the model to the IMDb sentiment classification task.

## 11. Optional Mini Fine-tuning Preview

This section is not the full Chapter 3 training lesson, but it gives students a preview.

For a real class, you can run this cell if Colab GPU is available.

Check GPU:

```text
Runtime → Change runtime type → T4 GPU
```

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import numpy as np
from transformers import Trainer, TrainingArguments

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./imdb_distilbert_demo",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

### Run training

This may take a few minutes on Colab GPU.

If it is too slow, reduce the training subset from 2000 to 500 examples.

In [ ]:
trainer.train()

## 12. Evaluate the Fine-tuned Model

In [ ]:
trainer.evaluate()

## 13. Try the Fine-tuned Model with a Pipeline

After training, we can use the model for prediction.

Because our model labels are numeric, we manually map them:

```text
LABEL_0 → Negative
LABEL_1 → Positive
```

In [ ]:
from transformers import pipeline

sentiment_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Positive"
}

def predict_sentiment(text):
    result = sentiment_classifier(text)[0]
    return {
        "label": label_map.get(result["label"], result["label"]),
        "score": round(result["score"], 4)
    }

predict_sentiment("This movie was wonderful. I loved the story and the acting.")

In [ ]:
predict_sentiment("The movie was boring, too long, and badly written.")

## 14. Classroom Discussion Questions

1. Why do we use a pretrained model instead of training from scratch?
2. What is the role of the tokenizer?
3. What is the role of the classification head?
4. Why is `distilbert-base-uncased` not already an IMDb classifier?
5. What changes during fine-tuning?
6. What would happen if our training dataset was very small?
7. What would happen if our training dataset had biased or noisy labels?

## 15. Student Exercise

Modify the notebook in one of the following ways:

### Easy

Change the test sentence and observe the prediction.

### Medium

Reduce the training data to 500 examples and compare the evaluation accuracy.

### Medium

Change `max_length` from 128 to 256 and observe memory/speed changes.

### Advanced

Try another dataset from the Hugging Face Hub, such as:

```python
load_dataset("yelp_polarity")
```

Then adapt the same workflow.

## 16. From This Notebook to the Rest of Chapter 3

This notebook corresponds to the introductory idea of Chapter 3:

```text
Fine-tuning a pretrained model
```

The full chapter continues with:

1. Processing data
2. Fine-tuning with the Trainer API
3. Writing a full training loop
4. Understanding learning curves
5. Using Accelerate for more flexible training

For your course, a clean teaching sequence is:

```text
Chapter 3.1: Why fine-tuning?
Chapter 3.2: Data processing
Chapter 3.3: Trainer API
Chapter 3.4: Custom training loop
Chapter 4: Share model to Hugging Face Hub
Chapter 9: Deploy model with Spaces
```

## 17. Key Takeaway

The most important sentence for students:

> Fine-tuning does not train a Transformer from zero. It starts from a pretrained Transformer and adapts it to a specific task using task-specific data.

For IMDb:

```text
Pretrained DistilBERT
+ IMDb reviews
+ sentiment labels
= IMDb sentiment classifier
```